# Robust regression

Least squares is pulled off course by a few extreme values. This first lecture of block 2 covers **robust regression**: Huber regression, which limits the influence of large residuals, and quantile regression, which models the median or another quantile instead of the mean. The second lecture of the block replaces the numeric target by a yes/no target ([07-lecture-logistic-regression.ipynb](07-lecture-logistic-regression.ipynb)).

```mermaid
flowchart TD
    Q{"What is the question?"} -->|"average price"| O["least squares:<br/>squared loss, mean"]
    Q -->|"typical price,<br/>a few gross errors"| H["Huber or median regression:<br/>large residuals count less"]
    Q -->|"a tail, e.g. the top 10 %"| T["quantile regression<br/>(q = 0.9)"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Robust regression: Huber

### Concept

Least squares squares the residuals, so a few gross errors (typing errors, unit mix-ups, placeholder values, a nightly price of €10,025) can pull the whole line towards them.

![Eight gross errors at large x tilt the least-squares line; the Huber line follows the bulk of the data](../figures/robust-vs-ols.png)

**Huber regression** uses a loss that is quadratic for small residuals and linear for large ones. Residuals smaller than a threshold ε (in units of a robust scale estimate; scikit-learn's default ε = 1.35) are treated as in least squares; larger residuals count only linearly, so a point 10 units away weighs 10 times, not 100 times, as much as a point 1 unit away. statsmodels' `RLM` fits the same loss by iteratively reweighted least squares and reports a **weight** per observation (1 = full weight, near 0 = treated as an outlier).

The **breakdown point** is the share of arbitrary outliers an estimator tolerates. It is 0 % for least squares: one point can move the line arbitrarily far.

### Why it matters

A robust fit shows what the bulk of the data says; comparing it with least squares reveals how much a few points drive the conclusion. This matters when outliers are errors or rare cases you do not want to model.

### How it works in Python

In [2]:
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import HuberRegressor, LinearRegression

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 80)
y = 2 + 0.5 * x + rng.normal(0, 0.5, 80)       # true slope 0.5
bad = np.argsort(x)[-8:]                       # 8 points with the largest x ...
y[bad] -= rng.uniform(6, 9, 8)                 # ... get gross errors
X = x.reshape(-1, 1)

print(round(LinearRegression().fit(X, y).coef_[0], 2))   # 0.07: pulled flat by 8 points
print(round(HuberRegressor().fit(X, y).coef_[0], 2))     # 0.41: close to the bulk

rlm = sm.RLM(y, sm.add_constant(x), M=sm.robust.norms.HuberT()).fit()
print(rlm.params.round(2))                     # [2.37 0.4 ]
print(rlm.weights[bad].mean().round(2), np.median(rlm.weights).round(2))   # 0.12 1.0: outliers down-weighted

0.07
0.41
[2.37 0.4 ]
0.12 1.0


The second line of the output shows the price of robustness: 0.41 instead of 0.5. With ε = 1.35 the 8 outliers still have some influence. Workbook [08](08-robust-fit-compared.ipynb) compares Huber with RANSAC and Theil–Sen, which tolerate more outliers.

### In practice

- RANSAC, a related robust method, was introduced for computer vision (Fischler & Bolles, 1981) and is still used to estimate camera geometry from image matches that contain many false matches.
- The Theil–Sen slope (Sen's slope) is a standard trend estimator in hydrology and climate research, often combined with the Mann–Kendall trend test.
- Sensor and laboratory data, in which occasional faulty readings are expected, are routinely fitted with M-estimators such as Huber's.

> **Warning.** **Outliers are information.** Down-weighting a point is a modelling decision. Inspect the points with low weights: are they errors, or the most important cases (fraud, failures, unusual products)? Report both fits if the conclusion depends on them.

> 📖 **Read more:** [extras/08-huber-vs-ridge.ipynb](extras/08-huber-vs-ridge.ipynb) (scikit-learn gallery): fits `HuberRegressor` with increasing ε on data with strong outliers, and shows that the Huber fit approaches the ridge fit, which follows the outliers, as ε grows.
>
> 📖 **Read more:** [extras/06-statsmodels-robust-models-intro.ipynb](extras/06-statsmodels-robust-models-intro.ipynb) (statsmodels examples): `RLM` with Huber's T and Andrew's wave norms on the stack-loss data, and OLS and RLM side by side on data with outliers.

> 📖 **Read more:** [extras/09-statsmodels-m-estimators.ipynb](extras/09-statsmodels-m-estimators.ipynb) (statsmodels examples): plots the loss and weight functions of seven M-estimators (Huber, Tukey's biweight, Hampel and others), explains robust scale estimates (MAD, IQR, Qn) and shows that M-estimators fail at high-leverage points.
>
> 📖 **Read more:** [extras/11-ransac.ipynb](extras/11-ransac.ipynb) (scikit-learn gallery): RANSAC fits the model on random subsets again and again and splits the data into inliers and outliers, so it copes with a large share of gross errors.
>
> 📖 **Read more:** [extras/12-theil-sen.ipynb](extras/12-theil-sen.ipynb) (scikit-learn gallery): the Theil–Sen estimator (a median of slopes, breakdown point about 29 %) against least squares and RANSAC, with outliers in y and in x.

## Robust regression: quantile regression

### Concept

Least squares models the **conditional mean** of y given x. **Quantile regression** models a **conditional quantile**: q = 0.5 gives the **median** line, q = 0.9 the line below which 90 % of observations lie. It minimises the sum of absolute residuals weighted by q above the line and by 1 − q below it (the **pinball loss**). The median line is therefore robust to outliers in y.

Fitting several quantiles shows whether x affects the whole distribution in the same way. Diverging slopes mean the spread of y changes with x (heteroscedasticity), which least squares summarises only as a violation of an assumption.

Worked example: for the values 1, 2, 3, 4, 100, the mean is 22 and the median 3. A constant "model" fitted by least squares predicts 22; one fitted by median regression predicts 3.

### Why it matters

Many decisions concern a typical case or a tail rather than the average: delivery-time promises, capacity planning, risk limits. Quantile regression answers "what happens at the 90th percentile?" directly and needs no assumption about the residual distribution.

### How it works in Python

The listings contain a real version of the gross errors in the figure. A house for five and a loft for seven guests ask €7,999 and €10,025 a night, 15 km from the centre (Session 4 flagged them; they may be typing errors or prices that block bookings). Other high prices are genuine: houseboats for 12 to 16 guests in Treptow-Köpenick, about 16 to 22 km out, ask €1,800 to €4,500. Fit the price in euros (not on the log scale, so that the outliers act fully) on the number of guests and the distance to Alexanderplatz. A host-pricing use case adds a second question: what do the **top 10 %** of comparable listings charge?

In [3]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()
short["km_to_centre"] = 111.2 * np.hypot(short["latitude"] - 52.5219,
                                         (short["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
train, test = train_test_split(short, test_size=0.2, random_state=42)
print(train.nlargest(2, "price")[["price", "accommodates", "km_to_centre"]].round(1).to_dict("records"))

formula = "price ~ accommodates + km_to_centre"                    # raw EUR, so the outliers act fully
fits = {"OLS (mean)": smf.ols(formula, train).fit(),
        "median (q=0.5)": smf.quantreg(formula, train).fit(q=0.5),
        "q=0.9": smf.quantreg(formula, train).fit(q=0.9),
        "OLS, prices < 1,000": smf.ols(formula, train[train["price"] < 1000]).fit()}
for name, fit in fits.items():
    pred = fit.predict(test)
    print(f"{name:20s} intercept {fit.params['Intercept']:6.1f}  per guest {fit.params['accommodates']:5.1f}"
          f"  per km {fit.params['km_to_centre']:5.2f}  test MAE {mean_absolute_error(test['price'], pred):5.1f}"
          f"  share of test prices below {np.mean(test['price'] <= pred):.2f}")
huber = HuberRegressor(max_iter=1000).fit(train[["accommodates", "km_to_centre"]], train["price"])
print("Huber                intercept", round(huber.intercept_, 1), " per guest", round(huber.coef_[0], 1),
      " per km", round(huber.coef_[1], 2))

[{'price': 10025.0, 'accommodates': 7, 'km_to_centre': 14.5}, {'price': 7999.2, 'accommodates': 5, 'km_to_centre': 15.6}]


OLS (mean)           intercept   59.6  per guest  38.3  per km -0.61  test MAE  70.7  share of test prices below 0.61
median (q=0.5)       intercept   76.7  per guest  30.7  per km -3.29  test MAE  65.5  share of test prices below 0.49
q=0.9                intercept  117.3  per guest  54.5  per km -4.64  test MAE 123.6  share of test prices below 0.89
OLS, prices < 1,000  intercept   93.7  per guest  30.9  per km -3.59  test MAE  66.9  share of test prices below 0.60
Huber                intercept 81.4  per guest 30.6  per km -3.37


```
[{'price': 10025.0, 'accommodates': 7, 'km_to_centre': 14.5}, {'price': 7999.2, 'accommodates': 5, 'km_to_centre': 15.6}]
OLS (mean)           intercept   59.6  per guest  38.3  per km -0.61  test MAE  70.7  share of test prices below 0.61
median (q=0.5)       intercept   76.7  per guest  30.7  per km -3.29  test MAE  65.5  share of test prices below 0.49
q=0.9                intercept  117.3  per guest  54.5  per km -4.64  test MAE 123.6  share of test prices below 0.89
OLS, prices < 1,000  intercept   93.7  per guest  30.9  per km -3.59  test MAE  66.9  share of test prices below 0.60
Huber                intercept 81.4  per guest 30.6  per km -3.37
```

The extreme prices flatten the least-squares distance effect to €0.61 per km: in the OLS world, location hardly matters. The two most extreme prices alone do most of the damage, because both lie about 15 km from the centre where few listings are: dropping just these two rows moves the OLS slope to €2.64 per km. The median line (€3.29 per km), the Huber line (€3.37) and least squares without the 25 training prices above €1,000 (€3.59) agree: a night costs about €3.30 to €3.60 less per kilometre from Alexanderplatz, and about €31 more per additional guest. The robust fits also predict better (test MAE €65.5 against €70.7). The 90th-percentile line answers the host's other question: at the upper end of the market, each guest adds €54 and each kilometre costs €4.64; 89 % of the test prices lie below it, close to the intended 90 %. A host who wants to price "like the better listings nearby" would use that line; one who wants a typical price, the median line. Removing or correcting the extreme prices is still the right fix once they are confirmed as errors (Session 4); the robust fits show what happens before you have found them.

### In practice

- Paediatric growth charts report percentiles of height and weight by age; quantile regression is one method used to construct such reference curves (Wei et al., 2006).
- Financial risk: Value at Risk is a quantile of the loss distribution; CAViaR models (Engle & Manganelli, 2004) estimate it with quantile regression.
- Engel's law: quantile regression on Engel's 1857 household data, a standard example since Koenker and Bassett (1982), shows that food expenditure rises with income at every quantile, but more steeply at the upper quantiles ([extras/10-quantile-regression-statsmodels.ipynb](extras/10-quantile-regression-statsmodels.ipynb), statsmodels examples).

> 📖 **Read more:** [extras/07-quantile-regression-sklearn.ipynb](extras/07-quantile-regression-sklearn.ipynb) (scikit-learn gallery): `QuantileRegressor` on data with heteroscedastic and with asymmetric noise, and why the median fit and least squares then differ.

> **Important.** **Worked example.** Section 2.1 of [04-case-study-airbnb-price-and-churn.ipynb](04-case-study-airbnb-price-and-churn.ipynb) compares least squares with Huber and quantile regression for the nightly price, with and without the extreme prices. Which model would you use to suggest a typical price to a new host, and which to tell them what the most expensive 10 % of comparable listings charge? Add room type to the model: do the conclusions change?

> **Caution.** Quantile lines fitted separately can cross (the 0.9 line below the 0.5 line for some x), especially at the edges of the data. Check a plot before reporting them.

## Check your understanding

1. What does the Huber loss do with a residual of 10 compared with least squares?
2. Why does a handful of extreme prices flatten the least-squares distance effect from about €3.50 to €0.61 per km, while the median line is not affected? What question does the 90th-percentile line answer?
3. About which share of the test prices should lie below the line of a quantile regression with q = 0.9, and how do you check it on the test data?

## Further reading

- Koenker, R., & Hallock, K. F. (2001). Quantile regression. *Journal of Economic Perspectives*, 15(4), 143–156. <https://doi.org/10.1257/jep.15.4.143>
- Huber, P. J., & Ronchetti, E. M. (2009). *Robust Statistics* (2nd ed.). Wiley.